# Epic vs. Tragedy — better comparisons

In [14]:
import numpy as np
import pandas as pd

homer_df = pd.read_parquet("../parquet/homer_dtm.parquet")
tragedy_df = pd.read_parquet("../parquet/tragedy-with-years_dtm.parquet")

In [15]:
homer_series = homer_df["iliad"] + homer_df["odyssey"]
tragedy_series = tragedy_df.sum(axis=1)

homer_series, tragedy_series = homer_series.align(tragedy_series, fill_value=0)

total_homer = homer_series.sum()
total_tragedy = tragedy_series.sum()

# apply Laplace smoothing
V = len(homer_series)
alpha = 1
p_epic = (homer_series + alpha) / (total_homer + alpha * V)
p_tragedy = (tragedy_series + alpha) / (total_tragedy + alpha * V)
log_ratio = np.log(p_epic / p_tragedy)

In [85]:
log_ratio.sort_values().to_csv("../csv/epic_tragedy_log_ratio.csv")

In [28]:
tragedy_aligned = tragedy_df.reindex(log_ratio.index, fill_value=0)
speaker_score = tragedy_aligned.multiply(log_ratio, axis=0).sum(axis=0) / tragedy_df.sum(axis=0)

In [29]:
speaker_score.name = "epicness"
speaker_df = speaker_score.reset_index()

In [30]:
speaker_df["token_count"] = tragedy_aligned.sum(axis=0).values

In [44]:
play_df = speaker_df.groupby(["dramatist", "title"]).apply(
    lambda g: (g["epicness"] * g["token_count"]).sum() / g["token_count"].sum()).reset_index().rename(
        columns={0: "epicness"}
    )

In [52]:
import altair as alt

alt.Chart(play_df).mark_point().encode(
    x="epicness:Q",
    y=alt.Y("dramatist:N", sort=["Aeschylus", "Sophocles", "Euripides"]),
    color=alt.Color("dramatist:N", sort=["Aeschylus", "Sophocles", "Euripides"]),
    tooltip=["dramatist:N", "title:N", "epicness:Q"],
).properties(title="Epicness by play and dramatist")

alt.Chart(...)

In [75]:
speaker_df["label"] = speaker_df["dramatist"] + " – " + speaker_df["title"] + " – " + speaker_df["speaker"]

filtered_speaker_df = speaker_df[speaker_df["token_count"] >= 50]
speaker_df_viz = pd.concat([filtered_speaker_df.nsmallest(20, "epicness"), filtered_speaker_df.nlargest(20, "epicness")])
alt.Chart(speaker_df_viz).mark_point().encode(
    x="epicness:Q",
    y=alt.Y("label:N", sort="x", axis=alt.Axis(labelLimit=400)),
    color=alt.Color("dramatist:N", sort=["Aeschylus", "Sophocles", "Euripides"]),
    tooltip=["speaker:N", "dramatist:N", "title:N", "epicness:Q"],
).properties(title="Epicness by speaker and play", width=400)

alt.Chart(...)

In [78]:
speaker_df.to_parquet("../parquet/epicness_by_speaker.parquet")
log_ratio.name = "epicness"
log_ratio.reset_index().to_parquet("../parquet/epicness_log_ratio.parquet")

In [ ]:
line_epicness_df = pd.read_parquet("../parquet/tragedy_line_epicness.parquet")

line_epicness_df["heat"] = line_epicness_df["line_epicness"]

# Per-tragedy aggregate: mean of the per-line scores
tragedy_heat = (
    line_epicness_df.groupby(["dramatist", "title"])["line_epicness"]
    .mean()
    .sort_values()
)
line_epicness_df["line_urn"] = line_epicness_df["urn"] + ":" + line_epicness_df["n"].astype(str)
line_epicness_df["heat_scaled"] = line_epicness_df["heat"] / line_epicness_df["heat"].abs().max() * 10
line_epicness_df[["line_urn", "heat", "heat_scaled"]].sort_values(by="heat", ascending=True).to_csv("../csv/tragedy_urn_heat.csv")